# SpatialVLA Confidence Analysis

**목표**: SpatialVLA가 gripper → target object 접근 시 confidence(action token probability)가 올라가는지 검증한다.

**방법**:
- Bridge v2 (WidowX) 데이터셋 에피소드에서 각 observation 이미지마다 SpatialVLA inference
- `generate(output_scores=True)`로 각 action token의 softmax probability 추출
- **confidence = Σ log P(translation token, rotation token, gripper token)**
- 로봇 EEF Z좌표 vs confidence 상관관계 계산

**환경 요구사항**: Python 3.12 OK · SimplerEnv 불필요 · GPU(CUDA) 필요

## Cell 0: 의존성 설치

In [ ]:
# transformers는 정확히 4.47.0으로 고정 (4.50+에서 _validate_images_text_input_order 제거됨)
!pip install "transformers==4.47.0" --upgrade -q
!pip install torch torchvision pillow matplotlib scipy -q
!pip install tensorflow tensorflow-datasets -q
print("설치 완료 — Runtime → Restart session 후 Cell 1부터 다시 실행")

## Cell 1: SpatialVLA 모델 로드

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoModel, AutoProcessor

MODEL_ID   = "IPEC-COMMUNITY/spatialvla-4b-224-sft-bridge"
UNNORM_KEY = "bridge_orig/1.0.0"   # bridge fine-tuned 모델의 action 정규화 키
TASK_TEXT  = "put the spoon on the towel"  # Bridge dataset task 중 하나

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16
).eval().cuda()

print("Model loaded.")

## Cell 1-b: 단일 이미지 sanity check

SpatialVLA는 **3개의 3D 격자 토큰**으로 1-step action을 출력합니다:

| 토큰 | 의미 | 내부 구조 |
|------|------|-----------|
| `translation_grid` | 이동 방향 (x,y,z) | 3D 격자 인덱스 1개 |
| `rotation_grid`    | 회전 방향 (roll,pitch,yaw) | 3D 격자 인덱스 1개 |
| `gripper`          | 그리퍼 open/close | 이진 토큰 1개 |

→ `max_new_tokens=3` (토큰 7개 쓰면 4~7번째는 다음 step action 토큰을 침범함)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image

# ── 테스트 이미지 로드 ────────────────────────────────────────────────────
# 직접 보유한 이미지 경로로 교체 가능 (Bridge-style 로봇 사진 권장)
IMAGE_PATH = "SpatialVLA/test/example.png"
TEST_TASK  = "pick up the egg"

image = Image.open(IMAGE_PATH).convert("RGB")

# ── 입력 준비 ─────────────────────────────────────────────────────────────
inputs = processor(
    images=[image],
    text=TEST_TASK,
    unnorm_key=UNNORM_KEY,
    return_tensors="pt"
).to(model.device)
inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

# ── 추론: max_new_tokens=3 (translation_grid · rotation_grid · gripper) ──
# ※ 7로 설정하면 4~7번째 토큰은 다음 step의 action 토큰을 침범함
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        return_dict_in_generate=True,
        output_scores=True,
        do_sample=False
    )

input_len    = inputs["input_ids"].shape[1]
gen_tokens   = outputs.sequences[0, input_len:]   # 생성된 토큰 3개

# ── 토큰별 log P 계산 ─────────────────────────────────────────────────────
# SpatialVLA 3-token 구조에 맞는 이름
TOKEN_NAMES = ["translation_grid", "rotation_grid", "gripper      "]

print(f"\n{'Token':<18} {'log P':>8} {'P%':>8}   bar")
print("─" * 60)

log_probs = []
for name, score, tok in zip(TOKEN_NAMES, outputs.scores, gen_tokens):
    lp = F.log_softmax(score[0].float(), dim=-1)[tok].item()
    p  = np.exp(lp)
    log_probs.append(lp)
    bar = "█" * int(p * 40)
    print(f"{name:<18} {lp:>8.3f} {p:>7.1%}   {bar}")

print("─" * 60)
print(f"{'Σ log P':<18} {sum(log_probs):>8.3f}   ← confidence")

# ── action 디코딩: 3토큰 → (dx,dy,dz, droll,dpitch,dyaw, gripper) 7D ─────
# processor.decode_actions가 내부적으로 3D 격자 → 연속값 변환을 처리함
action_out  = processor.decode_actions(
    outputs.sequences[:, input_len:], unnorm_key=UNNORM_KEY
)
pred_action = action_out["actions"][0]   # shape (7,)

labels = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]
print(f"\nDecoded action (7D EEF delta):")
for lbl, val in zip(labels, pred_action):
    print(f"  {lbl:<8} = {val:+.4f}")


## Cell 2: Bridge v2 에피소드 로드

Bridge v2 데이터셋 (WidowX robot)을 GCS 퍼블릭 버킷에서 직접 스트리밍합니다.
별도 인증 없이 Colab에서 접근 가능합니다.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

GCS_PATH = "gs://gresearch/robotics/bridge_orig/1.0.0/"

# GCS 접근 실패 시: !gcloud auth application-default login
builder = tfds.builder_from_directory(GCS_PATH)
ds = builder.as_dataset(split='train')

# 성공 에피소드 1개 찾기 (마지막 step reward > 0)
episode_steps = None
for episode in ds.take(50):  # 최대 50개 에피소드 탐색
    steps = list(episode['steps'])
    last_reward = float(steps[-1]['reward'].numpy())
    if last_reward > 0:  # 성공 에피소드
        episode_steps = steps
        print(f"성공 에피소드 발견: {len(steps)} steps")
        break

if episode_steps is None:
    print("성공 에피소드 미발견 → 첫 번째 에피소드 사용")
    episode_steps = list(list(ds.take(1))[0]['steps'])
    print(f"에피소드: {len(episode_steps)} steps")

# 첫 step의 observation key 확인
print("Observation keys:", list(episode_steps[0]['observation'].keys()))

# 첫 프레임 미리보기
obs0 = episode_steps[0]['observation']
img_key = 'image_0' if 'image_0' in obs0 else 'image'
plt.figure(figsize=(5, 4))
plt.imshow(obs0[img_key].numpy())
plt.title("Episode first frame")
plt.axis("off")
plt.show()

## Cell 3: 프레임별 inference + confidence 추출

각 step의 observation 이미지에 SpatialVLA를 실행하고,
3개 action token (translation / rotation / gripper)의 log probability를 합산합니다.

- `output_scores=True` → 각 생성 step의 logits 반환
- `F.log_softmax(...)[token_id]` → 해당 token의 log probability
- **confidence = Σ log P = joint log-probability of the 3 action tokens**

In [ ]:
history = []

for step_idx, step in enumerate(episode_steps):
    obs = step['observation']

    # 이미지 추출
    img_key = 'image_0' if 'image_0' in obs else 'image'
    img_array = obs[img_key].numpy()  # (H, W, 3) uint8
    image = Image.fromarray(img_array)

    # 로봇 EEF state (x, y, z, roll, pitch, yaw, gripper)
    state = obs['state'].numpy() if 'state' in obs else None

    # SpatialVLA 입력 준비
    inputs = processor(
        images=[image],
        text=TASK_TEXT,
        unnorm_key=UNNORM_KEY,
        return_tensors="pt"
    ).to(model.device)
    # pixel_values dtype 맞춤 (processor가 float32로 반환하므로 bfloat16으로 변환)
    inputs = {
        k: v.to(torch.bfloat16) if torch.is_floating_point(v) else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        # output_scores=True: 각 생성 step의 logits를 scores로 반환
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,              # action token 3개만 생성
            return_dict_in_generate=True,
            output_scores=True,
            do_sample=False                # greedy decoding
        )

    input_len = inputs["input_ids"].shape[1]
    gen_tokens = outputs.sequences[0, input_len:]  # 생성된 3개 토큰 ID

    # 각 토큰의 log P
    # outputs.scores: tuple of (batch, vocab_size) tensors, one per generated token
    lps = [
        F.log_softmax(s[0].float(), dim=-1)[t].item()
        for s, t in zip(outputs.scores, gen_tokens)
    ]
    conf = sum(lps)              # Σ log P (joint log-probability)
    probs = [np.exp(lp) for lp in lps]   # 각 토큰의 softmax probability

    # action 디코딩 (gripper open/close 확인용)
    action_out = processor.decode_actions(
        outputs.sequences[:, input_len:], unnorm_key=UNNORM_KEY
    )
    pred_action = action_out["actions"][0]  # (7,): [dx,dy,dz, droll,dpitch,dyaw, gripper]

    history.append({
        "step": step_idx,
        "conf": conf,
        "lps": lps,
        "probs": probs,
        "state": state,        # EEF pose (x,y,z,roll,pitch,yaw,gripper)
        "pred_action": pred_action,
        "reward": float(step['reward'].numpy()),
        "image": img_array,
    })

    state_str = f" | EEF_z={state[2]:.3f}" if state is not None else ""
    print(
        f"[{step_idx:3d}] ΣlogP={conf:7.2f} | "
        f"P_trans={probs[0]:.1%} P_rot={probs[1]:.1%} P_grip={probs[2]:.1%}"
        + state_str
    )

print(f"\n완료: {len(history)} steps")

## Cell 4: 시각화 + 상관분석

핵심 질문: **EEF Z ↓ (gripper 내려감) 와 confidence ↑ 가 상관이 있는가?**

- `Pearson r(EEF_z, confidence) < 0` → gripper 내려갈수록 confidence 증가 = 가설 지지
- EEF Z가 없으면 step 번호를 시간 proxy로 사용

In [ ]:
from scipy.stats import pearsonr, spearmanr

steps  = [h["step"] for h in history]
confs  = [h["conf"] for h in history]
p_t    = [h["probs"][0] for h in history]   # translation token P
p_r    = [h["probs"][1] for h in history]   # rotation token P
p_g    = [h["probs"][2] for h in history]   # gripper token P
grips  = [h["pred_action"][-1] for h in history]  # predicted gripper command

# EEF Z 좌표 (있으면)
eef_z     = [h["state"][2] if h["state"] is not None else None for h in history]
has_state = (eef_z[0] is not None)

def smooth(arr, w=5):
    """이동 평균 스무딩"""
    return np.convolve(arr, np.ones(w) / w, mode='same')

ncols = 3 if has_state else 2
fig, axes = plt.subplots(2, ncols, figsize=(6 * ncols, 9))

# ── (1) Σ log P over time ──────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(steps, confs, alpha=0.3, color='steelblue', label='raw')
ax.plot(steps, smooth(confs), color='steelblue', linewidth=2, label='smoothed')
ax.set_title("Σ log P(action tokens) over steps\n(높을수록 confident)", fontsize=11)
ax.set_xlabel("Step"); ax.set_ylabel("Σ log P")
ax.legend(); ax.grid(True)

# ── (2) Per-token probability ──────────────────────────────────────────────
ax = axes[0, 1]
for vals, name, c in zip(
    [p_t, p_r, p_g],
    ["translation", "rotation", "gripper"],
    ["tomato", "seagreen", "royalblue"]
):
    ax.plot(steps, smooth(vals), color=c, linewidth=1.5, label=name)
ax.set_title("Per-token softmax probability over steps", fontsize=11)
ax.set_xlabel("Step"); ax.set_ylabel("P")
ax.legend(); ax.grid(True)

# ── (3) EEF Z vs Confidence scatter (핵심 그래프) ─────────────────────────
if has_state:
    ax = axes[0, 2]
    sc = ax.scatter(eef_z, confs, c=steps, cmap='viridis', s=25, alpha=0.8)
    plt.colorbar(sc, ax=ax, label="Step")
    ax.set_title("Confidence vs. EEF Z\n(Z 낮을수록 gripper가 물체 근처)", fontsize=11)
    ax.set_xlabel("EEF Z (m)"); ax.set_ylabel("Σ log P")

# ── (4) EEF Z over time ───────────────────────────────────────────────────
ax = axes[1, 0]
if has_state:
    ax.plot(steps, eef_z, color='darkorange', linewidth=2)
    ax.set_title("EEF Z over time (↓ = gripper 내려감)", fontsize=11)
    ax.set_xlabel("Step"); ax.set_ylabel("Z (m)"); ax.grid(True)
else:
    ax.text(0.5, 0.5, "EEF state not available",
            ha='center', va='center', transform=ax.transAxes)

# ── (5) Gripper command over time ─────────────────────────────────────────
ax = axes[1, 1]
ax.step(steps, grips, color='seagreen', where='mid', linewidth=2)
ax.set_title("Predicted gripper command\n(양수=open, 음수=close)", fontsize=11)
ax.set_xlabel("Step"); ax.set_ylabel("Gripper cmd"); ax.grid(True)

# ── (6) 마지막 프레임 ─────────────────────────────────────────────────────
if has_state:
    ax = axes[1, 2]
else:
    ax = axes[1, 1]  # 상태 없으면 gripper 그래프 자리에 표시
ax.imshow(history[-1]["image"])
ax.set_title(f"Final observation (step {history[-1]['step']}, reward={history[-1]['reward']})",
             fontsize=10)
ax.axis("off")

plt.suptitle(f"SpatialVLA Confidence Analysis\nTask: '{TASK_TEXT}'", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── 상관 분석 ────────────────────────────────────────────────────────────
print("=" * 50)
print("상관 분석 결과")
print("=" * 50)

r_step, p_step = pearsonr(steps, confs)
print(f"step vs Σ log P  → Pearson r = {r_step:+.3f}  (p = {p_step:.4f})")

if has_state:
    r_z, p_z = pearsonr(eef_z, confs)
    r_z_sp, p_z_sp = spearmanr(eef_z, confs)
    print(f"EEF_z vs Σ log P → Pearson  r = {r_z:+.3f}  (p = {p_z:.4f})")
    print(f"                   Spearman r = {r_z_sp:+.3f}  (p = {p_z_sp:.4f})")
    print()
    if r_z < -0.3 and p_z < 0.05:
        print("✓ r_z < 0  →  gripper 내려갈수록(target 접근) confidence ↑  [가설 지지]")
    elif r_z > 0.3 and p_z < 0.05:
        print("✗ r_z > 0  →  gripper 올라갈수록 confidence ↑  [가설 반박]")
    else:
        print("? 뚜렷한 상관 없음 (|r| < 0.3 또는 p ≥ 0.05)")